# Mesh algorithms

**Algorithms:** `compute_vertex_normals`, `compute_mean_curvature`, `closest_point`, `cast_ray`, `compute_mesh_summary` from `zarr_vectors_tools.algorithms`

All five operate on a chunked `mesh` store. The two attribute algorithms stream chunk by chunk and accumulate into a global per-vertex array. The two spatial queries use the existing `chunks_intersecting_bbox` helper / a 3D-DDA chunk walk to localise candidate triangles. The summary streams per-face area and signed-tetrahedron volume.

A unit sphere has closed-form ground truth for every output, so we can spot-check the numbers.

1. Build a UV-sphere and write the mesh
2. `compute_vertex_normals` — compare to analytic outward normals
3. `compute_mean_curvature` — compare to `1 / R` analytic
4. `closest_point` from an exterior probe
5. `cast_ray` through the centre — should hit at `t ≈ R - origin_to_centre_dist`
6. `compute_mesh_summary` — surface area, volume, Euler characteristic

In [1]:
import numpy as np, tempfile, os
from zarr_vectors.lazy import open_zvr
from zarr_vectors.types.meshes import write_mesh
from zarr_vectors_tools.algorithms import (
    cast_ray,
    closest_point,
    compute_mean_curvature,
    compute_mesh_summary,
    compute_vertex_normals,
)

_tmpdir = tempfile.mkdtemp(prefix="zvt_examples_")
STORE = os.path.join(_tmpdir, "sphere.zarrvectors")
print("Store:", STORE)

Store: C:\Users\andre\AppData\Local\Temp\zvt_examples_tqm_3oi1\sphere.zarrvectors


## 1 · Build a UV-sphere and write the mesh

Radius 50 µm centred at (150, 150, 150). `chunk_shape=(50, 50, 50)` gives an 8-chunk grid over the sphere's bounding cube, so plenty of triangles straddle chunk boundaries — useful for exercising the cross-chunk accounting.

In [2]:
RADIUS = 50.0
CENTRE = np.array([150., 150., 150.], dtype=np.float64)

def uv_sphere(radius, lat_steps=60, lon_steps=120, offset=(0., 0., 0.)):
    verts = []
    for i in range(lat_steps + 1):
        theta = np.pi * i / lat_steps
        for j in range(lon_steps):
            phi = 2 * np.pi * j / lon_steps
            verts.append([
                radius * np.sin(theta) * np.cos(phi) + offset[0],
                radius * np.sin(theta) * np.sin(phi) + offset[1],
                radius * np.cos(theta)               + offset[2],
            ])
    verts = np.array(verts, dtype=np.float32)

    faces = []
    for i in range(lat_steps):
        for j in range(lon_steps):
            a = i * lon_steps + j
            b = a + lon_steps
            c = a + 1 if j < lon_steps - 1 else i * lon_steps
            d = b + 1 if j < lon_steps - 1 else (i + 1) * lon_steps
            if i > 0:
                faces.append([a, b, c])
            if i < lat_steps - 1:
                faces.append([b, d, c])
    return verts, np.array(faces, dtype=np.int32)

vertices, faces = uv_sphere(
    radius=RADIUS, lat_steps=60, lon_steps=120, offset=tuple(CENTRE),
)
print(f"vertices : {vertices.shape}")
print(f"faces    : {faces.shape}")

write_mesh(
    STORE,
    vertices=vertices,
    faces=faces,
    chunk_shape=(50., 50., 50.),
    bin_shape=(12.5, 12.5, 12.5),
)
store = open_zvr(STORE)
print(f"\nvertex_count : {store[0].vertex_count:,}")
print(f"chunk_shape  : {store.chunk_shape}")

vertices : (7320, 3)
faces    : (14160, 3)

vertex_count : 7,320
chunk_shape  : (50.0, 50.0, 50.0)


## 2 · Per-vertex normals

On a sphere centred at `C`, the outward unit normal at vertex `v` is `(v - C) / R`. The area-weighted result returned by `compute_vertex_normals` should match this to within a small angular error that grows near chunk boundaries (where intra-chunk faces alone are summed).

In [3]:
out = compute_vertex_normals(STORE, weighting="area")
n_computed = out["normals"]
print(f"computed normals       : {n_computed.shape}, dtype={n_computed.dtype}")
print(f"incomplete boundary    : {out['incomplete_boundary_vertices']}")

# `compute_vertex_normals` returns normals in the global vertex ordering,
# which corresponds to read_mesh()'s output order. Re-read positions in
# the same order so we can compare against analytic ground truth.
from zarr_vectors.types.meshes import read_mesh
m = read_mesh(STORE)
P = m["vertices"].astype(np.float64)
n_analytic = (P - CENTRE) / np.linalg.norm(P - CENTRE, axis=1, keepdims=True)

cos = np.clip(np.einsum("ij,ij->i", n_computed.astype(np.float64), n_analytic), -1, 1)
angle_deg = np.degrees(np.arccos(cos))
print(f"\nAngular error vs analytic (degrees):")
print(f"  median : {np.median(angle_deg):.3f}")
print(f"  p95    : {np.percentile(angle_deg, 95):.3f}")
print(f"  max    : {angle_deg.max():.3f}")

computed normals       : (7320, 3), dtype=float32
incomplete boundary    : 1102

Angular error vs analytic (degrees):
  median : 0.046
  p95    : 1.580
  max    : 90.000


## 3 · Mean curvature

For a sphere of radius `R`, the mean curvature is `1/R` everywhere. The cotangent-Laplace–Beltrami estimator should land close, with boundary vertices (those touching cross-chunk edges) biased low because faces straddling chunks aren't contributed.

In [4]:
curv = compute_mean_curvature(STORE)
kappa = curv["mean_curvature"]
expected = 1.0 / RADIUS

print(f"expected (1/R)         : {expected:.4f}")
print(f"computed median        : {np.median(kappa):.4f}")
print(f"computed p5  / p95     : {np.percentile(kappa, 5):.4f}  /  {np.percentile(kappa, 95):.4f}")
print(f"incomplete boundary    : {curv['incomplete_boundary_vertices']}")

rel = (kappa - expected) / expected
print(f"\nrelative error  median={np.median(rel):+.3f}   p95(|rel|)={np.percentile(np.abs(rel), 95):.3f}")

expected (1/R)         : 0.0200
computed median        : 0.0200
computed p5  / p95     : 0.0200  /  0.4329
incomplete boundary    : 1102

relative error  median=+0.000   p95(|rel|)=20.647


## 4 · Closest point on the mesh

Probe from an exterior point along the `+x` axis. The closest point on a sphere lies on the line from the centre through the probe, at distance `R` from the centre. The expected hit distance is `‖query - centre‖ - R`.

In [5]:
query = CENTRE + np.array([80., 0., 0.])     # 30 µm outside the surface
expected_dist = float(np.linalg.norm(query - CENTRE) - RADIUS)
expected_point = CENTRE + (query - CENTRE) * RADIUS / np.linalg.norm(query - CENTRE)

cp = closest_point(STORE, query=query)
print(f"found        : {cp['found']}")
print(f"distance     : {cp['distance']:.4f}    (expected {expected_dist:.4f})")
print(f"position     : {cp['position'].round(3)}")
print(f"  expected   : {expected_point.round(3)}")
print(f"chunk_key    : {cp['chunk_key']}")
print(f"face_index   : {cp['face_index']}")

# Cap the tolerance at the UV-sphere's facet size (a couple of µm).
assert cp['found']
assert abs(cp['distance'] - expected_dist) < 2.0

found        : True
distance     : 30.1822    (expected 30.0000)
position     : [199.931 150.    147.383]
  expected   : [200. 150. 150.]
chunk_key    : (3, 3, 2)
face_index   : 0


## 5 · Ray casting

Shoot a ray from an external origin towards the sphere centre. The first hit should land on the near surface, with `t = ‖origin - centre‖ - R`.

In [6]:
# Diagonal ray: origin in the +x+y+z corner, direction toward the
# centre. The entry point lands on the sphere at a generic location
# in the interior of chunk (3, 3, 3), well away from any chunk
# boundary. An axis-aligned ray (e.g. along -x) would enter at the
# +x pole, which sits exactly on a chunk boundary and is a
# *cross-chunk* triangle — cast_ray v0 doesn't test those, so it would
# walk straight through to the back surface. See the docstring in
# mesh_query.py for the limitation.
diag = np.array([1., 1., 1.]) / np.sqrt(3.)
origin    = CENTRE + 200.0 * diag      # 200 µm from centre
direction = -diag                      # points at the sphere centre
expected_t = float(np.linalg.norm(origin - CENTRE) - RADIUS)

ray = cast_ray(STORE, origin=origin, direction=direction)
print(f"hit        : {ray['hit']}")
print(f"t          : {ray['t']:.4f}    (expected {expected_t:.4f})")
print(f"position   : {ray['position'].round(3)}")
print(f"  on sphere?: |pos - centre| = {np.linalg.norm(ray['position'] - CENTRE):.4f}  (R = {RADIUS})")
print(f"chunk_key  : {ray['chunk_key']}")
print(f"face_index : {ray['face_index']}")

assert ray['hit']
assert abs(ray['t'] - expected_t) < 2.0

# A ray that misses the sphere — fire it sideways from the same origin.
miss = cast_ray(STORE, origin=origin, direction=np.array([0., 0., 1.]))
print(f"\nperpendicular ray hit = {miss['hit']}  (expected False)")

hit        : True
t          : 150.0127    (expected 150.0000)
position   : [178.86 178.86 178.86]
  on sphere?: |pos - centre| = 49.9873  (R = 50.0)
chunk_key  : (3, 3, 3)
face_index : 1049

perpendicular ray hit = False  (expected False)


## 6 · Streaming mesh summary

Surface area, volume, and the Euler characteristic. For a closed sphere with consistent winding:

- Surface area = `4πR² ≈ 31_416 µm²` (for R = 50)
- Volume      = `4/3 πR³ ≈ 523_599 µm³`
- Euler `χ`   = `2`

Cross-chunk *faces* lose identity in current core storage and aren't contributed; the summary reports the excluded edge count so you can quantify the gap. Per-object summaries (`per_object=True`) need core *Add 6* and currently raise `NotImplementedError`.

In [7]:
summary = compute_mesh_summary(STORE)

exp_area = 4 * np.pi * RADIUS ** 2
exp_vol  = (4 / 3) * np.pi * RADIUS ** 3

print(f"surface_area : {summary['surface_area']:.2f}    (analytic 4πR² = {exp_area:.2f})")
print(f"volume       : {abs(summary['volume']):.2f}    (analytic 4/3·πR³ = {exp_vol:.2f})")
print(f"face_count   : {summary['face_count']:,}")
print(f"vertex_count : {summary['vertex_count']:,}")
print(f"edge_count   : {summary['edge_count']:,}")
print(f"euler χ      : {summary['euler_characteristic']}    (closed surface expects 2)")
print(f"excluded cross-chunk face edges: {summary['excluded_cross_face_edges']}")

# `compute_mesh_summary(per_object=True)` is reserved for catalog Add 6.
try:
    compute_mesh_summary(STORE, per_object=True)
except NotImplementedError as exc:
    print(f"\nper_object → NotImplementedError, as documented: {exc}")

surface_area : 29510.58    (analytic 4πR² = 31415.93)
volume       : 492843.77    (analytic 4/3·πR³ = 523598.78)
face_count   : 13,251
vertex_count : 7,320
edge_count   : 21,480
euler χ      : -909    (closed surface expects 2)
excluded cross-chunk face edges: 2727

per_object → NotImplementedError, as documented: per_object summaries require core support for face-to-object mapping and `object_attributes` on write_mesh. Tracked as catalog Add 6.


## Summary

| Step | API |
|------|-----|
| Vertex normals | `compute_vertex_normals(store, weighting='area' | 'uniform')` |
| Mean curvature | `compute_mean_curvature(store)` |
| Closest point  | `closest_point(store, query, max_distance=None, max_expansion_rings=4)` |
| Ray cast       | `cast_ray(store, origin, direction, max_distance=None)` |
| Summary stats  | `compute_mesh_summary(store)` |

Known v0 limitations (documented inline in each algorithm's docstring):

- Boundary vertices in normals / curvature receive intra-chunk faces only — the returned `incomplete_boundary_vertices` count quantifies how many vertices are affected.
- Cross-chunk faces aren't tested in `closest_point` / `cast_ray`; for typical meshes this is a tiny minority of faces, but a small probe lying *exactly* on a chunk-spanning triangle could miss.
- `write_back=True` on normals/curvature, and `per_object=True` on the summary, both raise `NotImplementedError` pending core *Add 5* and *Add 6*.